# Example 1.3.1 / 2: Binary Variables

Companion notebook for **1.3.1 Integer, Binary and Restricted Variables** in the AMPL lab Documentation, section "Binary Variables". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Binary Variables

`binary` is shorthand for `integer >= 0, <= 1`. A binary variable models a yes-or-no decision,
usually linked to a continuous variable by a constraint. Here a product can only be made if
its line is set up, and setting up a line has a fixed cost.

`data.py` gives several parameters at once from one pandas DataFrame: `ampl.set_data(df, "PROD")`
sets the index as the members of `PROD` and each column as the parameter of the same name. In
`run.py`, `ampl.get_data("Use, Make")` returns both variables in one table, and `to_pandas()`
turns it into a DataFrame. The solver's values can be off by tiny round-off amounts such as
`-3.8e-16`, so the table is rounded before printing:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param setup {PROD} >= 0;     # fixed cost of setting up the line
param market {PROD} >= 0;

var Make {p in PROD} >= 0;
var Use {PROD} binary;       # 1 if the line for product p is set up

maximize Net_Profit: sum {p in PROD} (profit[p] * Make[p] - setup[p] * Use[p]);
subject to Only_If_Set_Up {p in PROD}: Make[p] <= market[p] * Use[p];
subject to Capacity: sum {p in PROD} Make[p] <= 5000;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

df = pd.DataFrame(
    {"profit": [25, 30, 29], "setup": [40000, 80000, 10000], "market": [6000, 4000, 3500]},
    index=["bands", "coils", "plate"],
)
ampl.set_data(df, "PROD")

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
df = ampl.get_data("Use, Make").to_pandas()
print(df.round(6) + 0)   # tiny round-off values (and -0.0) shown as 0

HiGHS 1.14.0: optimal solution; objective 91500
3 simplex iterations
1 branching nodes
solved
       Use    Make
bands  0.0     0.0
coils  0.0     0.0
plate  1.0  3500.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `option display_eps` makes `display` show tiny
round-off values as 0, as the rounding does in `run.py`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param:  profit  setup   market :=
  bands    25    40000   6000
  coils    30    80000   4000
  plate    29    10000   3500 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
option display_eps 1e-6;   # show tiny round-off values as 0
display Use, Make;

HiGHS 1.14.0: optimal solution; objective 91500
3 simplex iterations
1 branching nodes
:     Use   Make    :=
bands   0      0
coils   0      0
plate   1   3500
;

